In [1]:
import os
import json
import random
from openai import OpenAI
from IPython.display import Markdown, display
from scraper import fetch_products_by_category, CATEGORY_URLS

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "phi3"

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

try:
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Reply with just: OK"}],
    )
    print("Connected to Ollama. Model responded:", test.choices[0].message.content.strip())
except Exception as e:
    print("Could not reach Ollama at", OLLAMA_BASE_URL)
    print("Make sure `ollama serve` is running and you've run `ollama pull phi3`.")
    print("Error:", e)

Connected to Ollama. Model responded: OK


In [3]:
_USER_DB = {
    "u_1001": {
        "user_id": "u_1001",
        "name": "Amina",
        "age": 29,
        "recent_views": ["laptop for university", "lightweight notebook"],
        "purchase_history": ["laptop sleeve", "wireless mouse"],
        "budget_hint": "mid-range",
        "location": "Cairo, Egypt",
    },
    "u_1002": {
        "user_id": "u_1002",
        "name": "Karim",
        "age": 41,
        "recent_views": ["new phone", "flagship smartphone"],
        "purchase_history": ["phone case", "screen protector"],
        "budget_hint": "premium",
        "location": "Giza, Egypt",
    },
}

def fetch_user_profile(user_id: str) -> dict:
    """MOCK API CALL #1 — simulates GET /users/{user_id}/profile"""
    print(f"[Mock API 1] GET /users/{user_id}/profile")
    if user_id not in _USER_DB:
        raise ValueError(f"No such user: {user_id}")
    return _USER_DB[user_id]

In [4]:
user_profile = fetch_user_profile("u_1001")
user_profile

[Mock API 1] GET /users/u_1001/profile


{'user_id': 'u_1001',
 'name': 'Amina',
 'age': 29,
 'recent_views': ['laptop for university', 'lightweight notebook'],
 'purchase_history': ['laptop sleeve', 'wireless mouse'],
 'budget_hint': 'mid-range',
 'location': 'Cairo, Egypt'}

In [5]:
intent_system_prompt = """
You are a shopping intent analysis engine for an e-commerce recommendation system.
You will be given a user's recent views and purchase history.
The store only sells three categories: "laptops", "tablets", "phones".
Pick the ONE category that best matches what the user wants next, and
estimate a maximum price in USD they'd likely want to spend.

Respond with ONLY a JSON object, no explanation, no markdown code fences,
in exactly this format:

{"category": "laptops", "max_price": 600, "reasoning": "one short sentence"}

category must be exactly one of: "laptops", "tablets", "phones".
"""

In [6]:
def get_intent_user_prompt(user_profile: dict) -> str:
    return f"""
Here is the user's shopping activity:

Recently viewed: {", ".join(user_profile["recent_views"])}
Past purchases: {", ".join(user_profile["purchase_history"])}
Stated budget preference: {user_profile["budget_hint"]}

Pick the single best category and a max price. Respond with only the JSON object.
"""

print(get_intent_user_prompt(user_profile))


Here is the user's shopping activity:

Recently viewed: laptop for university, lightweight notebook
Past purchases: laptop sleeve, wireless mouse
Stated budget preference: mid-range

Pick the single best category and a max price. Respond with only the JSON object.



In [7]:
def _extract_json(text: str) -> dict:
    """Small local models sometimes wrap JSON in text or code fences — clean it up."""
    text = text.strip()
    if "```" in text:
        text = text.split("```")[1]
        text = text.replace("json", "", 1).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model output: {text!r}")
    return json.loads(text[start:end + 1])


def analyze_user_intent(user_profile: dict) -> dict:
    """LOCAL LLM CALL #1 (phi3 via Ollama) — turns behavior data into a category + budget"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": intent_system_prompt},
            {"role": "user", "content": get_intent_user_prompt(user_profile)},
        ],
        temperature=0.2,
    )
    raw = response.choices[0].message.content
    try:
        result = _extract_json(raw)
    except (ValueError, json.JSONDecodeError):
        print("Could not parse model output as JSON, using fallback.")
        print("Raw output was:", raw)
        result = {"category": "laptops", "max_price": 800, "reasoning": "fallback default"}

    if result.get("category") not in CATEGORY_URLS:
        print(f"Model picked an invalid category {result.get('category')!r}, defaulting to 'laptops'.")
        result["category"] = "laptops"
    return result

In [8]:
search_intent = analyze_user_intent(user_profile)
search_intent

{'category': 'laptops', 'max_price': 800}

In [9]:
def search_live_catalog(intent: dict, max_results: int = 5) -> list:
    """REAL WEB FETCH — hits the live site, no mocking involved"""
    category = intent["category"]
    max_price = intent.get("max_price")
    print(f"[Real fetch] GET {CATEGORY_URLS[category]}")

    all_products = fetch_products_by_category(category)
    print(f"  -> parsed {len(all_products)} live products from the page")

    if max_price:
        in_budget = [p for p in all_products if p["price"] is not None and p["price"] <= max_price]
        # if nothing fits the budget, fall back to the full list rather than showing nothing
        candidates = in_budget or all_products
    else:
        candidates = all_products

    candidates.sort(key=lambda p: p["price"] if p["price"] is not None else float("inf"))
    return candidates[:max_results]

In [10]:
candidate_products = search_live_catalog(search_intent)
candidate_products

[Real fetch] GET https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops
  -> parsed 117 live products from the page


[{'name': 'Asus VivoBook X441NA-GA190',
  'price': 295.99,
  'description': 'Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd',
  'reviews': 14,
  'url': 'https://webscraper.io/test-sites/e-commerce/allinone/product/60'},
 {'name': 'Prestigio SmartBook 133S Dark Grey',
  'price': 299.0,
  'description': 'Prestigio SmartBook 133S Dark Grey, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam',
  'reviews': 8,
  'url': 'https://webscraper.io/test-sites/e-commerce/allinone/product/61'},
 {'name': 'Prestigio SmartBook 133S Gold',
  'price': 299.0,
  'description': 'Prestigio SmartBook 133S Gold, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam',
  'reviews': 12,
  'url': 'https://webscraper.io/test-sites/e-commerce/allinone/product/62'},
 {'name': 'Aspire E1-510',
  'price': 306.99,
  'description': '15.6", Pentium N3520 2.16GHz, 4GB, 500GB, Linux',
  'reviews': 2,
  'url':

In [11]:
def check_inventory_and_pricing(products: list) -> dict:
    """MOCK API CALL #2 — simulates GET /inventory?urls=..."""
    urls = [p["url"] for p in products]
    print(f"[Mock API 2] GET /inventory?count={len(urls)}")
    random.seed(42)
    inventory = {}
    for url in urls:
        in_stock = random.random() > 0.15
        discount = random.choice([0, 0, 0, 10, 15])
        inventory[url] = {
            "in_stock": in_stock,
            "stock_count": random.randint(1, 40) if in_stock else 0,
            "discount_percent": discount,
        }
    return inventory

In [12]:
inventory_info = check_inventory_and_pricing(candidate_products)
inventory_info

[Mock API 2] GET /inventory?count=5


{'https://webscraper.io/test-sites/e-commerce/allinone/product/60': {'in_stock': True,
  'stock_count': 18,
  'discount_percent': 0},
 'https://webscraper.io/test-sites/e-commerce/allinone/product/61': {'in_stock': True,
  'stock_count': 7,
  'discount_percent': 0},
 'https://webscraper.io/test-sites/e-commerce/allinone/product/62': {'in_stock': True,
  'stock_count': 6,
  'discount_percent': 15},
 'https://webscraper.io/test-sites/e-commerce/allinone/product/32': {'in_stock': True,
  'stock_count': 2,
  'discount_percent': 0},
 'https://webscraper.io/test-sites/e-commerce/allinone/product/63': {'in_stock': False,
  'stock_count': 0,
  'discount_percent': 0}}

In [13]:
recommendation_system_prompt = """
You are a friendly personal shopping assistant writing a short product
recommendation message directly to a customer.
Only recommend items that are in stock. Mention discounts if any exist.
Keep it warm, concise, and specific (2-4 short paragraphs). Respond in
plain markdown text, no code fences.
"""

In [14]:
def get_recommendation_user_prompt(user_profile, products, inventory) -> str:
    lines = [f"Customer name: {user_profile['name']}", "", "Live shortlisted products:"]
    for p in products:
        inv = inventory[p["url"]]
        status = f"IN STOCK ({inv['stock_count']} left)" if inv["in_stock"] else "OUT OF STOCK"
        discount = f", {inv['discount_percent']}% off" if inv["discount_percent"] else ""
        price = f"${p['price']:.2f}" if p["price"] is not None else "price unavailable"
        lines.append(f"- {p['name']} — {price}{discount} — {status} — {p['reviews']} reviews")
    lines.append("")
    lines.append("Write the personalized recommendation message now.")
    return "\n".join(lines)

print(get_recommendation_user_prompt(user_profile, candidate_products, inventory_info))

Customer name: Amina

Live shortlisted products:
- Asus VivoBook X441NA-GA190 — $295.99 — IN STOCK (18 left) — 14 reviews
- Prestigio SmartBook 133S Dark Grey — $299.00 — IN STOCK (7 left) — 8 reviews
- Prestigio SmartBook 133S Gold — $299.00, 15% off — IN STOCK (6 left) — 12 reviews
- Aspire E1-510 — $306.99 — IN STOCK (2 left) — 2 reviews
- Lenovo V110-15IAP — $321.94 — OUT OF STOCK — 5 reviews

Write the personalized recommendation message now.


In [15]:
def generate_recommendations(user_profile, products, inventory):
    """LOCAL LLM CALL #2 (phi3 via Ollama) — final personalized message using all prior outputs"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": recommendation_system_prompt},
            {"role": "user", "content": get_recommendation_user_prompt(user_profile, products, inventory)},
        ],
        temperature=0.7,
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
generate_recommendations(user_profile, candidate_products, inventory_info)

Hi Amina,


I hope this message finds you well! I've been reviewing the options available for a new laptop and wanted to share some exciting choices that align with your needs for a reliable and stylish device.


First, I noticed the Asus VivoBook X441NA-GA190, which is priced at $295.99 and currently has 18 left in stock. Customers have given it a 4.5-star rating based on 18 reviews, and what they've been saying about its performance and design has been overwhthy.


Moving on to the Prestigio SmartBook series, there are two models that caught my eye: the SmartBook 133S Dark Grey at $299.00 and the Gold version of the same model, which is also $299.00 but currently has a 15% discount bringing the price to $254.15. Both models are stocked, with the Dark Grey at 7 left and the Gold at 6 left, and users have given them an impressive 4.2 and 4.0-star ratings, respectively, based on 8 and 12 reviews.


Lastly, I wanted to mention the Aspire E1-510, another excellent choice at $306.99 with only 2 left in stock. It has received a respectable 3.8-star rating from its 2 reviews.


I hope these recommendations help you in finding the perfect laptop! If you have any questions or need further assistance, I'm here to help.


Warm regards,


[Your Name]

Personal Shopping Assistant